# Hardware Benchmarking: PyTorch CPU vs GPU vs TPU

This notebook evaluates the computational performance of a trained 1D CNN for Human Activity Recognition across different hardware accelerators. We compare training and inference execution time, throughput and available hardware information using PyTorch.

## Import libraries

In [1]:
import sys
import time
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from pathlib import Path
from torch.utils.data import Dataset, DataLoader, Subset

print("Python version:", platform.python_version())
print("Operating system:", platform.system())
print("PyTorch version:", torch.__version__)


Python version: 3.11.18
Operating system: Windows
PyTorch version: 2.15.0+cpu


## Configure project paths

In [2]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from paths import (
    DATA_DIR,
    MODEL_DIR,
    RESULTS_DIR
)

PROCESSED_DATA_FILE = DATA_DIR / "processed" / "har_preprocessed.pt"
MODEL_FILE = MODEL_DIR / "har_cnn_baseline.pth"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed dataset exists:", PROCESSED_DATA_FILE.exists())
print("Model checkpoint exists:", MODEL_FILE.exists())


Project root: d:\sensor-har-pytorch-accelerators
Processed dataset exists: True
Model checkpoint exists: True


## Detect available hardware

In [3]:
print("PyTorch version:", torch.__version__)

print("\nCPU:")
print("Logical CPU cores:", __import__("os").cpu_count())

print("\nCUDA:")
print("Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nXLA / TPU:")
try:
    import torch_xla
    print("PyTorch/XLA version:", torch_xla.__version__)
except ImportError:
    print("PyTorch/XLA is not installed.")


PyTorch version: 2.15.0+cpu

CPU:
Logical CPU cores: 16

CUDA:
Available: False

XLA / TPU:
PyTorch/XLA is not installed.


## Load the dataset and model

In [4]:
# Load the preprocessed dataset
data = torch.load(
    PROCESSED_DATA_FILE,
    map_location="cpu",
    weights_only=False
)

print("Dataset keys:", data.keys())

# Load the trained model checkpoint
checkpoint = torch.load(
    MODEL_FILE,
    map_location="cpu",
    weights_only=True
)

print("\nCheckpoint keys:", checkpoint.keys())
print("Architecture:", checkpoint.get("architecture", "Not specified"))
print("Number of classes:", checkpoint.get("num_classes", "Not specified"))
print("Input channels:", checkpoint.get("input_channels", "Not specified"))

Dataset keys: dict_keys(['X_train', 'X_test', 'y_train', 'y_test', 'train_indices', 'val_indices'])

Checkpoint keys: dict_keys(['model_state_dict', 'architecture', 'num_classes', 'input_channels', 'sequence_length', 'batch_size', 'epochs', 'test_loss', 'test_accuracy'])
Architecture: HAR_CNN
Number of classes: 6
Input channels: 6


## Reconstruct the model

In [5]:
class HAR_CNN(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()

        self.conv1 = nn.Conv1d(6, 32, kernel_size=5, padding=2)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 32, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)

        return x


# Reconstruct the model and load its trained weights
model = HAR_CNN(num_classes=checkpoint["num_classes"])
model.load_state_dict(checkpoint["model_state_dict"])

# Use evaluation mode for inference benchmarking
model.eval()

# Count trainable parameters
total_parameters = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Model loaded successfully.")
print("Architecture:", checkpoint["architecture"])
print("Trainable parameters:", f"{total_parameters:,}")
print("Model mode:", "Evaluation" if not model.training else "Training")

Model loaded successfully.
Architecture: HAR_CNN
Trainable parameters: 274,342
Model mode: Evaluation


## Prepare the test data

In [7]:
# Reuse the tensors loaded from the preprocessed dataset
X_test_tensor = data["X_test"].detach().clone().to(torch.float32)
y_test_tensor = data["y_test"].detach().clone().to(torch.long)

# Create DataLoader
BATCH_SIZE = 64

test_dataset = torch.utils.data.TensorDataset(
    X_test_tensor,
    y_test_tensor
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Inspect the data
signals, labels = next(iter(test_loader))

print("Test samples:", len(test_dataset))
print("Number of batches:", len(test_loader))
print("Batch size:", BATCH_SIZE)
print("Signal batch shape:", signals.shape)
print("Label batch shape:", labels.shape)
print("Signal data type:", signals.dtype)
print("Label data type:", labels.dtype)

Test samples: 2947
Number of batches: 47
Batch size: 64
Signal batch shape: torch.Size([64, 6, 128])
Label batch shape: torch.Size([64])
Signal data type: torch.float32
Label data type: torch.int64


## Benchmark inference on CPU

In [8]:
import time

# Select CPU
device = torch.device("cpu")
model = model.to(device)
model.eval()

# Benchmark configuration
WARMUP_BATCHES = 5
NUM_RUNS = 3

def benchmark_inference(model, data_loader, device, warmup_batches=5):

    model.eval()

    # Warm-up
    with torch.inference_mode():
        for i, (signals, _) in enumerate(data_loader):
            if i >= warmup_batches:
                break

            signals = signals.to(device)
            _ = model(signals)

    run_times = []

    # Actual benchmark
    for run in range(NUM_RUNS):

        start_time = time.perf_counter()

        with torch.inference_mode():
            for signals, _ in data_loader:
                signals = signals.to(device)
                _ = model(signals)

        end_time = time.perf_counter()

        elapsed_time = end_time - start_time
        run_times.append(elapsed_time)

        print(f"Run {run + 1}: {elapsed_time:.4f} seconds")

    # Calculate average performance
    average_time = np.mean(run_times)
    average_batch_time = average_time / len(data_loader)
    throughput = len(data_loader.dataset) / average_time

    return {
        "device": str(device),
        "average_time_seconds": average_time,
        "average_batch_time_ms": average_batch_time * 1000,
        "throughput_samples_per_second": throughput
    }


cpu_results = benchmark_inference(
    model,
    test_loader,
    device
)

print("\nCPU Inference Benchmark Results")
print("--------------------------------")
for key, value in cpu_results.items():
    print(f"{key}: {value}")

Run 1: 0.0851 seconds
Run 2: 0.0880 seconds
Run 3: 0.0924 seconds

CPU Inference Benchmark Results
--------------------------------
device: cpu
average_time_seconds: 0.08850053333056469
average_batch_time_ms: 1.8829900708630787
throughput_samples_per_second: 33299.23435593827


## Save the CPU benchmark results

In [9]:
# Store CPU benchmark results
cpu_results_df = pd.DataFrame([{
    **cpu_results,
    "cpu_logical_cores": 16,
    "pytorch_version": torch.__version__,
    "batch_size": BATCH_SIZE,
    "test_samples": len(test_dataset),
    "warmup_batches": WARMUP_BATCHES,
    "benchmark_runs": NUM_RUNS
}])

cpu_results_file = RESULTS_DIR / "inference_benchmark_cpu.csv"

cpu_results_df.to_csv(cpu_results_file, index=False)

print("CPU benchmark results saved to:", cpu_results_file)
display(cpu_results_df)

CPU benchmark results saved to: D:\sensor-har-pytorch-accelerators\results\inference_benchmark_cpu.csv


,device,average_time_seconds,average_batch_time_ms,throughput_samples_per_second,cpu_logical_cores,pytorch_version,batch_size,test_samples,warmup_batches,benchmark_runs
0,cpu,0.088501,1.88299,33299.234356,16,2.15.0+cpu,64,2947,5,3


## GPU benchmarking